# Modelo

In [1]:
import numpy as np
import pandas as pd
import joblib
import os

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import randint, uniform, norm, loguniform

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV

from config import SEED

Elegimos dataset

In [2]:
dataset = 'ciclos_c_r_hl256'

## Espectrogramas

Se suelen usar Mel espectrogramas

Traigo los consjuntos de entrenamiento y testeo. (TODO: buscar una manera más eficiente de guardarlos)

In [3]:
train_data = np.load(f'./dataset/{dataset}/train_melspectrogram.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/{dataset}/test_melspectrogram.npz')
X_test = test_data['X']
y_test = test_data['y']

In [4]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((9843, 47616), (9843,), (1296, 47616), (1296,))

### Random Forest

#### Entrenamiento

In [5]:
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, max_features='log2')

param_distributions = {
    'max_depth': randint(5, 10),
    'min_samples_split': randint(10, 25),
    'min_samples_leaf': randint(10, 25)
}

In [8]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END max_depth=8, min_samples_leaf=22, min_samples_split=24;, score=0.659 total time=  21.4s
[CV 2/5] END max_depth=8, min_samples_leaf=22, min_samples_split=24;, score=0.677 total time=  11.4s
[CV 3/5] END max_depth=8, min_samples_leaf=22, min_samples_split=24;, score=0.648 total time=   6.7s
[CV 4/5] END max_depth=8, min_samples_leaf=22, min_samples_split=24;, score=0.653 total time=   6.5s
[CV 5/5] END max_depth=8, min_samples_leaf=22, min_samples_split=24;, score=0.638 total time=   6.8s
[CV 1/5] END max_depth=7, min_samples_leaf=17, min_samples_split=22;, score=0.661 total time=   6.6s
[CV 2/5] END max_depth=7, min_samples_leaf=17, min_samples_split=22;, score=0.682 total time=   6.8s
[CV 3/5] END max_depth=7, min_samples_leaf=17, min_samples_split=22;, score=0.645 total time=   6.4s
[CV 4/5] END max_depth=7, min_samples_leaf=17, min_samples_split=22;, score=0.646 total time=   6.5s
[CV 5/5] END max_depth=7, min

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....0020E86C92BA0>, 'min_samples_leaf': <scipy.stats....0020E86CEC690>, 'min_samples_split': <scipy.stats....0020E86CEC410>}"
,n_iter,20
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [9]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(20)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
15,9,12,16,0.663876
2,9,16,19,0.663331
16,9,18,16,0.660860
10,9,21,18,0.660850
14,8,18,12,0.657120
18,8,23,11,0.656213
5,8,17,17,0.656118
0,8,22,24,0.655069
4,7,17,14,0.655029
1,7,17,22,0.655029


In [10]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 9, 'min_samples_leaf': 12, 'min_samples_split': 16}
Best CV score: 0.6638763726097423


In [11]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.77      0.73      0.75      5024
           1       0.73      0.78      0.75      4819

    accuracy                           0.75      9843
   macro avg       0.75      0.75      0.75      9843
weighted avg       0.75      0.75      0.75      9843



#### Evaluación

In [12]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.69      0.57      0.62       663
           1       0.62      0.73      0.67       633

    accuracy                           0.65      1296
   macro avg       0.65      0.65      0.65      1296
weighted avg       0.65      0.65      0.64      1296



#### Guardado

In [13]:
os.makedirs(f'./modelos/{dataset}', exist_ok=True)

joblib.dump(best_model, f'./modelos/{dataset}/melspec_rf.pkl')

['./modelos/ciclos_c_r_hl256/melspec_rf.pkl']

### Gradient Boost

In [19]:
gbc = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.01,
    subsample=0.8,
    n_iter_no_change=10,
    max_features='log2',
    max_depth=5,
    random_state=SEED
    )

In [20]:
gbc.fit(X_train, y_train)

,loss,'log_loss'
,learning_rate,0.01
,n_estimators,500
,subsample,0.8
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,5
,min_impurity_decrease,0.0
,init,None


In [21]:
gbc.n_estimators_

313

In [22]:
predict_train = gbc.predict(X_train)
print(classification_report(y_train, predict_train))

              precision    recall  f1-score   support

           0       0.73      0.70      0.72      5024
           1       0.70      0.74      0.72      4819

    accuracy                           0.72      9843
   macro avg       0.72      0.72      0.72      9843
weighted avg       0.72      0.72      0.72      9843



In [23]:
predict_test = gbc.predict(X_test)
print(classification_report(y_test, predict_test))

              precision    recall  f1-score   support

           0       0.68      0.59      0.63       663
           1       0.62      0.71      0.66       633

    accuracy                           0.64      1296
   macro avg       0.65      0.65      0.64      1296
weighted avg       0.65      0.64      0.64      1296



In [24]:
joblib.dump(gbc, f'./modelos/{dataset}/melspec_gbc.pkl')

['./modelos/ciclos_c_r_hl256/melspec_gbc.pkl']

## Features de Audio

In [25]:
train_data = np.load(f'./dataset/{dataset}/train_features.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/{dataset}/test_features.npz')
X_test = test_data['X']
y_test = test_data['y']

In [26]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((9843, 46), (9843,), (1296, 46), (1296,))

### Random Forest

#### Entrenamiento

In [35]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_distributions = {
    'max_depth': randint(10, 20),
    'min_samples_split': randint(10, 25),
    'min_samples_leaf': randint(10, 20)
}

In [36]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV 1/5] END max_depth=16, min_samples_leaf=13, min_samples_split=22;, score=0.728 total time=   0.8s
[CV 2/5] END max_depth=16, min_samples_leaf=13, min_samples_split=22;, score=0.769 total time=   0.8s
[CV 3/5] END max_depth=16, min_samples_leaf=13, min_samples_split=22;, score=0.760 total time=   0.8s
[CV 4/5] END max_depth=16, min_samples_leaf=13, min_samples_split=22;, score=0.753 total time=   0.8s
[CV 5/5] END max_depth=16, min_samples_leaf=13, min_samples_split=22;, score=0.742 total time=   0.9s
[CV 1/5] END max_depth=17, min_samples_leaf=14, min_samples_split=16;, score=0.729 total time=   0.8s
[CV 2/5] END max_depth=17, min_samples_leaf=14, min_samples_split=16;, score=0.768 total time=   0.8s
[CV 3/5] END max_depth=17, min_samples_leaf=14, min_samples_split=16;, score=0.753 total time=   0.8s
[CV 4/5] END max_depth=17, min_samples_leaf=14, min_samples_split=16;, score=0.755 total time=   0.8s
[CV 5/5] END max_dep

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....0020E86C57F00>, 'min_samples_leaf': <scipy.stats....0020EA66A5050>, 'min_samples_split': <scipy.stats....0020E86C57350>}"
,n_iter,50
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [37]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)#.head(20)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
9,18,10,20,0.754584
30,18,10,18,0.754584
10,19,12,21,0.752476
2,19,12,16,0.752476
15,18,11,19,0.751596
7,14,10,21,0.751265
14,16,11,13,0.750659
22,19,13,15,0.750539
11,16,13,18,0.750439
0,16,13,22,0.750439


In [38]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 18, 'min_samples_leaf': 10, 'min_samples_split': 20}
Best CV score: 0.7545835216748022


In [39]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.91      0.92      0.92      5024
           1       0.92      0.91      0.91      4819

    accuracy                           0.92      9843
   macro avg       0.92      0.92      0.92      9843
weighted avg       0.92      0.92      0.92      9843



#### Evaluación

In [40]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.75      0.77      0.76       663
           1       0.75      0.74      0.74       633

    accuracy                           0.75      1296
   macro avg       0.75      0.75      0.75      1296
weighted avg       0.75      0.75      0.75      1296



#### Guardado

In [41]:
joblib.dump(best_model, f'./modelos/{dataset}/features_rf.pkl')

['./modelos/ciclos_c_r_hl256/features_rf.pkl']